# Task 3 — Query Optimization: Caching

## Overview

This notebook tests caching using Queries 3a and 3b (air_quality_demand_by_band and air_quality_demand_correlation). Both queries independently compute the same borough_hour aggregation over integrated_taxi_trips before branching into their own logic. As written, this shared step runs twice, once per query.

Caching keeps a DataFrame's result in memory the first time it's computed, so later references to it skip recomputation. This is the only technique in this study whose benefit only shows up across multiple queries sharing a result, so this notebook times both queries together as a pair instead of timing one query alone.

## Query design

- Baseline: both queries exactly as written in the Task 1/2 library, each recomputing borough_hour independently.
- Optimized: borough_hour is computed once, cached with .cache(), and registered as a temp view (borough_hour_cached). Both downstream queries read from this view instead of recomputing the CTE. The banding and correlation logic itself is unchanged.

## Methodology

Because caching's benefit only shows up across multiple queries, we use timed_run_multi, which times both queries together as one unit per repetition, instead of timed_run, which repeats a single query. The cache is built once, outside the timed repetitions.

Order matters here in a way it didn't for the other three techniques. Spark's cache matches on the analyzed logical plan, not on a name, so a query can get served from cache even if it never references the cached view, as long as its plan matches. Because the baseline's inline CTE matches the cached plan exactly, running the baseline after the cache exists would silently serve it from cache too. So the baseline is fully run, timed, and its plan captured before the cache is created anywhere in this notebook.

As with the other notebooks, one warm-up run is discarded, five repetitions are timed, and the median is reported.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from data_intensive_computing.benchmark_helpers import (
    build_spark_session,
    register_delta_tables,
    timed_run_multi,
    capture_explain,
    verify_same_results,
    append_benchmark_record,
    find_project_root,
)

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "task3_benchmark_results.jsonl"

spark = build_spark_session(app_name="Task3-Caching")
register_delta_tables(spark, PROCESSED_DIR)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1cea9d01-17d0-4dff-8dbe-c71204cd66e7;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Registered taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Registered weather: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/weather
Registered air_quality: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/air_quality
Registered taxi_zones: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_zones
Registered integrated_taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips


26/09/17 22:12:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Baseline queries

Both queries exactly as written in the Task 1/2 library, each with its own
copy of the `borough_hour` CTE. Nothing has been cached yet in this session —
this must remain true until the baseline is fully measured below.

In [2]:
BASELINE_BAND_SQL = """
WITH borough_hour AS (
    SELECT
        pickup_borough,
        pickup_hour,
        AVG(air_quality_pm25_mean) AS pm25,
        COUNT(trip_id) AS hourly_trip_count
    FROM integrated_taxi_trips
    WHERE air_quality_match_status = 'borough_exact_hour'
      AND pickup_borough IS NOT NULL
      AND air_quality_pm25_mean IS NOT NULL
    GROUP BY pickup_borough, pickup_hour
), banded AS (
    SELECT
        *,
        CASE
            WHEN pm25 <= 5.0 THEN '01_very_low_0_5'
            WHEN pm25 <= 15.0 THEN '02_low_5_15'
            WHEN pm25 <= 35.0 THEN '03_elevated_15_35'
            ELSE '04_high_over_35'
        END AS pm25_band
    FROM borough_hour
)
SELECT
    pm25_band,
    COUNT(*) AS borough_hour_count,
    ROUND(AVG(pm25), 3) AS average_pm25,
    ROUND(AVG(hourly_trip_count), 2) AS average_hourly_demand,
    MIN(hourly_trip_count) AS minimum_hourly_demand,
    MAX(hourly_trip_count) AS maximum_hourly_demand
FROM banded
GROUP BY pm25_band
ORDER BY pm25_band
"""

BASELINE_CORR_SQL = """
WITH borough_hour AS (
    SELECT
        pickup_borough,
        pickup_hour,
        AVG(air_quality_pm25_mean) AS pm25,
        COUNT(trip_id) AS hourly_trip_count
    FROM integrated_taxi_trips
    WHERE air_quality_match_status = 'borough_exact_hour'
      AND pickup_borough IS NOT NULL
      AND air_quality_pm25_mean IS NOT NULL
    GROUP BY pickup_borough, pickup_hour
)
SELECT
    COALESCE(pickup_borough, 'All boroughs') AS pickup_borough,
    COUNT(*) AS matched_borough_hours,
    ROUND(CORR(pm25, hourly_trip_count), 4) AS pm25_demand_pearson_correlation
FROM borough_hour
GROUP BY GROUPING SETS ((pickup_borough), ())
ORDER BY pickup_borough
"""

## Baseline: execution time and plan

Measured now, before anything in this session has been cached.

In [3]:
baseline_timing = timed_run_multi(
    spark, "air_quality_pair_baseline", [BASELINE_BAND_SQL, BASELINE_CORR_SQL], repetitions=5
)
print("Baseline (borough_hour recomputed twice, nothing cached yet):", baseline_timing)

Baseline (borough_hour recomputed twice, nothing cached yet): {'query_name': 'air_quality_pair_baseline', 'row_counts': [4, 4], 'repetitions': 5, 'all_times_seconds': [0.6785, 0.608, 0.6934, 0.5242, 0.4995], 'median_seconds': 0.608, 'min_seconds': 0.4995, 'max_seconds': 0.6934}


In [4]:
baseline_band_plan = capture_explain(spark.sql(BASELINE_BAND_SQL), mode="formatted")
print("=" * 30, "BASELINE BAND PLAN", "=" * 30)
print(baseline_band_plan)

============================== BASELINE BAND PLAN ==============================
== Physical Plan ==
AdaptiveSparkPlan (13)
+- Sort (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Exchange (9)
            +- HashAggregate (8)
               +- Project (7)
                  +- HashAggregate (6)
                     +- Exchange (5)
                        +- HashAggregate (4)
                           +- Project (3)
                              +- Filter (2)
                                 +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [trip_id#3563, pickup_hour#3566, pickup_borough#3570, air_quality_pm25_mean#3586, air_quality_match_status#3592, pickup_year#3567, pickup_month#3568]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips]
PushedFilters: [IsNotNull(air_quality_match_status), EqualTo(air_quality_match_status

## Cache the shared intermediate

We cache the data now, after the baseline has been fully measured. `.cache()` marks the DataFrame to be cached, but Spark doesn't build it until an action runs against it. We call `.count()` right away so it gets built here, not during the first timed query.

In [5]:
BOROUGH_HOUR_SQL = """
SELECT
    pickup_borough,
    pickup_hour,
    AVG(air_quality_pm25_mean) AS pm25,
    COUNT(trip_id) AS hourly_trip_count
FROM integrated_taxi_trips
WHERE air_quality_match_status = 'borough_exact_hour'
  AND pickup_borough IS NOT NULL
  AND air_quality_pm25_mean IS NOT NULL
GROUP BY pickup_borough, pickup_hour
"""

borough_hour_df = spark.sql(BOROUGH_HOUR_SQL).cache()
row_count = borough_hour_df.count()  # forces materialization into cache
borough_hour_df.createOrReplaceTempView("borough_hour_cached")
print(f"Cached borough_hour: {row_count} rows")

Cached borough_hour: 6435 rows


## Optimized queries

Rewritten to read from `borough_hour_cached` instead of recomputing the CTE.
The banding and correlation logic are unchanged.

In [6]:
OPTIMIZED_BAND_SQL = """
WITH banded AS (
    SELECT
        *,
        CASE
            WHEN pm25 <= 5.0 THEN '01_very_low_0_5'
            WHEN pm25 <= 15.0 THEN '02_low_5_15'
            WHEN pm25 <= 35.0 THEN '03_elevated_15_35'
            ELSE '04_high_over_35'
        END AS pm25_band
    FROM borough_hour_cached
)
SELECT
    pm25_band,
    COUNT(*) AS borough_hour_count,
    ROUND(AVG(pm25), 3) AS average_pm25,
    ROUND(AVG(hourly_trip_count), 2) AS average_hourly_demand,
    MIN(hourly_trip_count) AS minimum_hourly_demand,
    MAX(hourly_trip_count) AS maximum_hourly_demand
FROM banded
GROUP BY pm25_band
ORDER BY pm25_band
"""

OPTIMIZED_CORR_SQL = """
SELECT
    COALESCE(pickup_borough, 'All boroughs') AS pickup_borough,
    COUNT(*) AS matched_borough_hours,
    ROUND(CORR(pm25, hourly_trip_count), 4) AS pm25_demand_pearson_correlation
FROM borough_hour_cached
GROUP BY GROUPING SETS ((pickup_borough), ())
ORDER BY pickup_borough
"""

## Optimized: execution time and plan

In [7]:
optimized_timing = timed_run_multi(
    spark, "air_quality_pair_cached", [OPTIMIZED_BAND_SQL, OPTIMIZED_CORR_SQL], repetitions=5
)
print("Optimized (borough_hour cached, reused twice):", optimized_timing)

Optimized (borough_hour cached, reused twice): {'query_name': 'air_quality_pair_cached', 'row_counts': [4, 4], 'repetitions': 5, 'all_times_seconds': [0.1115, 0.1116, 0.1037, 0.1012, 0.0983], 'median_seconds': 0.1037, 'min_seconds': 0.0983, 'max_seconds': 0.1116}


In [8]:
optimized_band_plan = capture_explain(spark.sql(OPTIMIZED_BAND_SQL), mode="formatted")
print("=" * 30, "OPTIMIZED BAND PLAN (reads borough_hour_cached)", "=" * 30)
print(optimized_band_plan)

============================== OPTIMIZED BAND PLAN (reads borough_hour_cached) ==============================
== Physical Plan ==
AdaptiveSparkPlan (24)
+- Sort (23)
   +- Exchange (22)
      +- HashAggregate (21)
         +- Exchange (20)
            +- HashAggregate (19)
               +- Project (18)
                  +- InMemoryTableScan (1)
                        +- InMemoryRelation (2)
                              +- AdaptiveSparkPlan (17)
                                 +- == Final Plan ==
                                    ResultQueryStage (11)
                                    +- * HashAggregate (10)
                                       +- ShuffleQueryStage (9), Statistics(sizeInBytes=1717.5 KiB, rowCount=3.14E+4)
                                          +- Exchange (8)
                                             +- * HashAggregate (7)
                                                +- * Project (6)
                                                   +- * Filter (5)
 

## Correctness verification

This runs after both plans have already been captured above. Re-running `BASELINE_BAND_SQL` at this point would itself be served from cache, since the cache now exists. That's fine for a correctness check, which only compares result rows, but it means this step can no longer be used for timing. The two downstream queries are checked separately, since caching changes two independent queries, band and correlation, not one before/after pair.

In [9]:
band_correctness = verify_same_results(spark, BASELINE_BAND_SQL, OPTIMIZED_BAND_SQL)
corr_correctness = verify_same_results(spark, BASELINE_CORR_SQL, OPTIMIZED_CORR_SQL)

print("Band query:", band_correctness)
print("Correlation query:", corr_correctness)

assert band_correctness["correctness_verified"], "Band query results differ — investigate before recording a result."
assert corr_correctness["correctness_verified"], "Correlation query results differ — investigate before recording a result."


Band query: {'correctness_verified': True, 'rows_only_in_before': 0, 'rows_only_in_after': 0}
Correlation query: {'correctness_verified': True, 'rows_only_in_before': 0, 'rows_only_in_after': 0}


## Benchmark record

In [10]:
speedup = round(baseline_timing["median_seconds"] / optimized_timing["median_seconds"], 2)

record = {
    "query_name": "air_quality_demand_by_band_and_correlation",
    "technique": "caching",
    "before_time_sec": baseline_timing["median_seconds"],
    "after_time_sec": optimized_timing["median_seconds"],
    "speedup": speedup,
    "plan_change_summary": (
        "Extracting the shared borough_hour CTE into a cached DataFrame and reading "
        "both downstream queries from it replaces the Scan parquet / Filter / "
        "HashAggregate chain over integrated_taxi_trips with an InMemoryTableScan in "
        "each downstream query's plan; the shared aggregation is computed once instead "
        "of once per query."
    ),
    "correctness_verified": band_correctness["correctness_verified"] and corr_correctness["correctness_verified"],
}

append_benchmark_record(record, RESULTS_PATH)
print(f"Speedup: {speedup}x — appended to {RESULTS_PATH}")

Speedup: 5.86x — appended to /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/artifacts/task3_benchmark_results.jsonl


## Discussion

Spark's cache matches on the analyzed logical plan, not on a name. The baseline queries here never reference the cached view, but if they ran after the cache existed, they would still be served from it, because their inline CTE is logically identical to what got cached. This is why the baseline in this notebook is fully measured before the cache is created.

Our first attempt got this wrong: we built the cache before measuring the baseline, so the baseline was accidentally served from cache too. That run gave a meaningless 1.15x. The corrected order gives the real result below.

The two air quality queries each independently recompute the exact same borough_hour aggregation. That duplication is the reason caching helps here.

Caching carries a risk the other three techniques don't: if the underlying table gets updated after caching, queries against the cached view keep returning the old data until it's manually refreshed or unpersisted.

Caching only pays off if the same result is actually reused more than once. If a query only needs an intermediate result once, caching it is pure overhead with nothing to offset it.

Result: 5.86x speedup. The optimized plan shows an InMemoryTableScan instead of rescanning and reaggregating from source.

In [11]:
borough_hour_df.unpersist()
spark.stop()